# Lesson 20: Classification and logistic regression · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 20

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: Prepare the data

Fill the missing avg_weekly_hours with the column's median, and add monthly_contract (Annual -> 0, Monthly -> 1). Then store the churn rate of monthly-contract subscribers in monthly_rate.

<details><summary>Hint</summary>

fillna(subs["avg_weekly_hours"].median()), map({"Annual": 0, "Monthly": 1}), then subs.groupby("monthly_contract")["churned"].mean()[1].

</details>

In [ ]:
import pandas as pd

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))

# 1. Fill missing hours with the median
subs["avg_weekly_hours"] = subs["avg_weekly_hours"].fillna(...)

# 2. Annual -> 0, Monthly -> 1
subs["monthly_contract"] = subs["contract"].map(...)

# 3. Churn rate where monthly_contract is 1
monthly_rate = ...
print(monthly_rate)

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "_s = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\nassert subs[\"avg_weekly_hours\"].notna().all(), \"Fill every missing avg_weekly_hours value\"\nassert abs(subs[\"avg_weekly_hours\"].sum() - _s[\"avg_weekly_hours\"].fillna(_s[\"avg_weekly_hours\"].median()).sum()) < 1e-6, \"Fill the gaps with the median of avg_weekly_hours\"\nassert (subs[\"monthly_contract\"] == (_s[\"contract\"] == \"Monthly\").astype(int)).all(), \"Map Annual to 0 and Monthly to 1\"\nassert monthly_rate is not Ellipsis and abs(float(monthly_rate) - _s.loc[_s[\"contract\"] == \"Monthly\", \"churned\"].mean()) < 1e-9, \"monthly_rate should be the mean of churned where monthly_contract is 1\"\n")

## Exercise 2: Your first classifier

The data is prepared and split for you. Fit a LogisticRegression(max_iter=1000) on the training set, store its test accuracy in acc and the 'nobody churns' baseline accuracy in baseline.

<details><summary>Hint</summary>

model.fit(X_train, y_train), then accuracy_score(y_test, model.predict(X_test)). (y_test == 0).mean() is the share who stayed.

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))
subs["avg_weekly_hours"] = subs["avg_weekly_hours"].fillna(subs["avg_weekly_hours"].median())
subs["monthly_contract"] = subs["contract"].map({"Annual": 0, "Monthly": 1})
features = ["age", "monthly_fee_usd", "tenure_months", "avg_weekly_hours",
            "devices", "support_tickets_90d", "auto_renew", "monthly_contract"]
X = subs[features]
y = subs["churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
# Fit on the training data, then score on the test data
acc = ...
# Share of test subscribers who did NOT churn
baseline = ...
print(acc, baseline)

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "from sklearn.model_selection import train_test_split as _tts\n_s = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"avg_weekly_hours\"] = _s[\"avg_weekly_hours\"].fillna(_s[\"avg_weekly_hours\"].median())\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_f = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\n_Xtr, _Xte, _ytr, _yte = _tts(_s[_f], _s[\"churned\"], test_size=0.2, random_state=42, stratify=_s[\"churned\"])\nfrom sklearn.linear_model import LogisticRegression as _LR\n_want = (_LR(max_iter=1000).fit(_Xtr, _ytr).predict(_Xte) == _yte.values).mean()\nassert isinstance(acc, float) and abs(acc - _want) < 1e-9, \"acc should be accuracy_score(y_test, model.predict(X_test)) after fitting on the training data\"\nassert baseline is not Ellipsis and abs(float(baseline) - (_yte == 0).mean()) < 1e-9, \"baseline should be (y_test == 0).mean()\"\n")

## Exercise 3: Probabilities, not just classes

Get the predicted probability of churn (class 1) for every test subscriber, then count how many are above 0.3. Store them in p_churn and n_risky.

<details><summary>Hint</summary>

model.classes_ is [0, 1], so the churn probability is predict_proba(X_test)[:, 1]. (p_churn > 0.3).sum() counts the Trues.

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))
subs["avg_weekly_hours"] = subs["avg_weekly_hours"].fillna(subs["avg_weekly_hours"].median())
subs["monthly_contract"] = subs["contract"].map({"Annual": 0, "Monthly": 1})
features = ["age", "monthly_fee_usd", "tenure_months", "avg_weekly_hours",
            "devices", "support_tickets_90d", "auto_renew", "monthly_contract"]
X = subs[features]
y = subs["churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)

# 1. Probability of churn (the second column of predict_proba)
p_churn = ...
# 2. How many test subscribers have p_churn above 0.3?
n_risky = ...
print(n_risky)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "from sklearn.model_selection import train_test_split as _tts\n_s = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"avg_weekly_hours\"] = _s[\"avg_weekly_hours\"].fillna(_s[\"avg_weekly_hours\"].median())\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_f = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\n_Xtr, _Xte, _ytr, _yte = _tts(_s[_f], _s[\"churned\"], test_size=0.2, random_state=42, stratify=_s[\"churned\"])\nimport numpy as _np\nfrom sklearn.linear_model import LogisticRegression as _LR\n_p = _LR(max_iter=1000).fit(_Xtr, _ytr).predict_proba(_Xte)[:, 1]\nassert p_churn is not Ellipsis and len(p_churn) == len(_Xte), \"p_churn should have one probability per test subscriber\"\nassert _np.allclose(_np.asarray(p_churn, dtype=float), _p, atol=1e-6), \"p_churn should be the second column (class 1, churned): predict_proba(X_test)[:, 1]\"\nassert n_risky is not Ellipsis and int(n_risky) == int((_p > 0.3).sum()), \"n_risky should be (p_churn > 0.3).sum()\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=20) and take the quiz.